# Chunking Strategies for RAG

This notebook demonstrates document chunking strategies used in RAG and Generative AI workflows.

**Topics:** fixed-size, overlap, token-based, sentence, paragraph, recursive splitting, metadata, and a basic retrieval experiment.

## 1. Why Chunking?

Chunking splits large documents into smaller meaningful pieces before embedding. A typical RAG flow is:

`Document → Chunking → Embeddings → Vector DB → Retrieval → LLM → Answer`

Good chunks preserve enough context for accurate retrieval without being unnecessarily large.

In [ ]:
!pip install -q transformers langchain-text-splitters sentence-transformers scikit-learn

In [ ]:
import re
import numpy as np
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Sample Document

The same document is used for every strategy so the results can be compared fairly.

In [ ]:
document = """
Python is a popular programming language used for web development, automation, data science, and machine learning. It has a large ecosystem of libraries and frameworks.

Machine learning is a branch of artificial intelligence where computers learn patterns from data. Common machine learning tasks include classification, regression, clustering, and recommendation.

Deep learning is a subset of machine learning that uses neural networks with multiple layers. Deep learning is widely used in computer vision, natural language processing, speech recognition, and generative AI.

Pandas is a Python library used for data manipulation and analysis. It provides DataFrame and Series data structures and supports filtering, grouping, merging, and handling missing values.

NumPy is a Python library for numerical computing. It provides multidimensional arrays, mathematical functions, vectorized operations, and linear algebra capabilities.

PyTorch is an open-source deep learning framework used to build and train neural networks. It supports tensors, automatic differentiation, GPU acceleration, and modern deep learning workflows.

Computer vision enables computers to understand images and videos. Applications include image classification, object detection, image segmentation, and optical character recognition.

Natural language processing enables computers to understand and generate human language. Applications include sentiment analysis, translation, summarization, question answering, and chatbots.
""".strip()
print(document)

## 3. Fixed-Size Chunking

**Why use it?** Simple and fast, but it can cut sentences or concepts at arbitrary boundaries.

In [ ]:
def fixed_size_chunking(text, chunk_size=250):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

fixed_chunks = fixed_size_chunking(document, 250)
for i, chunk in enumerate(fixed_chunks, 1):
    print(f"Chunk {i}:\n{chunk}\n{'-'*70}")

## 4. Fixed-Size Chunking with Overlap

**Why use it?** Repeating boundary text helps preserve context between neighboring chunks.

In [ ]:
def fixed_overlap_chunking(text, chunk_size=250, overlap=50):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

overlap_chunks = fixed_overlap_chunking(document, 250, 50)
for i, chunk in enumerate(overlap_chunks, 1):
    print(f"Chunk {i}:\n{chunk}\n{'-'*70}")

## 5. Token-Based Chunking

**Why use it?** LLMs use tokens, so token-based splitting gives more direct control over context size.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def token_chunking(text, tokenizer, chunk_size=80, overlap=15):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks, start = [], 0
    while start < len(tokens):
        chunk_tokens = tokens[start:start+chunk_size]
        chunks.append(tokenizer.decode(chunk_tokens, skip_special_tokens=True))
        start += chunk_size - overlap
    return chunks

token_chunks = token_chunking(document, tokenizer)
for i, chunk in enumerate(token_chunks, 1):
    print(f"Chunk {i}:\n{chunk}\n{'-'*70}")

## 6. Sentence-Based Chunking

**Why use it?** Keeps complete sentences together and usually preserves meaning better than arbitrary character cuts.

In [ ]:
def sentence_chunking(text, sentences_per_chunk=2):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [" ".join(sentences[i:i+sentences_per_chunk]) for i in range(0, len(sentences), sentences_per_chunk)]

sentence_chunks = sentence_chunking(document, 2)
for i, chunk in enumerate(sentence_chunks, 1):
    print(f"Chunk {i}:\n{chunk}\n{'-'*70}")

## 7. Paragraph-Based Chunking

**Why use it?** Preserves the document's paragraph structure and works well for well-formatted text.

In [ ]:
def paragraph_chunking(text):
    return [p.strip() for p in text.split('\n\n') if p.strip()]

paragraph_chunks = paragraph_chunking(document)
for i, chunk in enumerate(paragraph_chunks, 1):
    print(f"Chunk {i}:\n{chunk}\n{'-'*70}")

## 8. Recursive Text Splitting ⭐

**Why use it?** Tries larger natural boundaries first, then smaller separators if a chunk is still too large. This is a strong general-purpose RAG baseline.

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)
recursive_chunks = recursive_splitter.split_text(document)

for i, chunk in enumerate(recursive_chunks, 1):
    print(f"Chunk {i}:\n{chunk}\n{'-'*70}")

## 9. Compare Chunk Statistics

In [ ]:
strategies = {
    'Fixed Size': fixed_chunks,
    'Fixed + Overlap': overlap_chunks,
    'Token Based': token_chunks,
    'Sentence Based': sentence_chunks,
    'Paragraph Based': paragraph_chunks,
    'Recursive': recursive_chunks,
}

print(f"{'Strategy':<22} {'Chunks':>8} {'Avg chars':>12} {'Min':>8} {'Max':>8}")
print('-'*65)
for name, chunks in strategies.items():
    sizes = [len(c) for c in chunks]
    print(f"{name:<22} {len(chunks):>8} {np.mean(sizes):>12.1f} {min(sizes):>8} {max(sizes):>8}")

## 10. Chunk Metadata

Production RAG systems should keep source information with each chunk for traceability and filtering.

In [ ]:
metadata_chunks = [
    {
        'chunk_id': i,
        'text': chunk,
        'source': 'sample_document.txt',
        'strategy': 'recursive',
        'chunk_size': 250,
        'overlap': 50,
    }
    for i, chunk in enumerate(recursive_chunks, 1)
]

for item in metadata_chunks[:3]:
    print(item)

## 11. Basic Retrieval Experiment

Chunking should ultimately be judged by retrieval quality. We embed chunks and compare a query using cosine similarity.

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
query = 'Which library is used for numerical computing and multidimensional arrays?'

def retrieve_top_k(chunks, query, k=3):
    chunk_embeddings = embedding_model.encode(chunks, normalize_embeddings=True)
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i]), chunks[i]) for i in top_indices]

for name in ['Fixed Size', 'Recursive']:
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    for rank, (idx, score, chunk) in enumerate(retrieve_top_k(strategies[name], query), 1):
        print(f"\nRank {rank} | Similarity: {score:.4f}\n{chunk}")

## 12. Strategy Comparison

| Strategy | Context Preservation | Complexity | Typical RAG Use |
|---|---|---|---|
| Fixed-size | Low–Medium | Easy | Simple baseline |
| Fixed + overlap | Medium–High | Easy | General RAG |
| Token-based | Medium–High | Medium | Token-controlled workflows |
| Sentence-based | High | Medium | Natural-language documents |
| Paragraph-based | High | Easy | Structured documents |
| Recursive | High | Medium | Strong general-purpose baseline |

There is no universally best strategy. Evaluate chunking with your own retrieval questions and metrics such as Precision@K and Recall@K.

## 13. Key Takeaways

1. Chunking creates manageable retrieval units.
2. Chunk size controls how much context each chunk contains.
3. Overlap helps preserve boundary context.
4. Token-based chunking controls token usage.
5. Sentence and paragraph splitting preserve natural structure.
6. Recursive splitting is a strong general-purpose baseline.
7. Different document types may need different strategies.
8. Store metadata such as source, page/section, and chunk ID.
9. The best strategy should be selected using retrieval evaluation, not guesswork.